In [4]:
# Matar procesos de Chrome que hayan quedado colgados en el contenedor
!pkill -9 chrome
!pkill -9 chromedriver
print("Procesos antiguos eliminados.")

Procesos antiguos eliminados.


In [5]:
#Reinicio del Driver con "Evasión" (Celda de Configuración)
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import pandas as pd
import time

options = Options()
# --- PARÁMETROS DE ESTABILIDAD ---
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage") # Obligatorio en Docker
options.add_argument("--disable-gpu")
options.add_argument("--remote-debugging-port=9222")

# --- PARÁMETROS DE EVASIÓN (Para que no detecte el bot) ---
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36")

try:
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    # Comando para ocultar que es un bot
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
      "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    print("Navegador reiniciado con éxito.")
except Exception as e:
    print(f"Error al reiniciar: {e}")

Navegador reiniciado con éxito.


In [6]:
#Carga y Acción Manual (Celda de Navegación)
url_test = "https://www.tripadvisor.es/Hotel_Review-g187514-d1192534-Reviews-Hotel_Riu_Plaza_Espana-Madrid.html"
driver.get(url_test)

print("¡ATENCIÓN! Ve a la interfaz gráfica del Docker ahora:")
print("1. Acepta las Cookies manualmente si aparecen.")
print("2. Baja con el scroll hasta que veas las reseñas físicamente.")
print("3. Una vez las veas, regresa aquí y ejecuta la siguiente celda.")

¡ATENCIÓN! Ve a la interfaz gráfica del Docker ahora:
1. Acepta las Cookies manualmente si aparecen.
2. Baja con el scroll hasta que veas las reseñas físicamente.
3. Una vez las veas, regresa aquí y ejecuta la siguiente celda.


In [6]:
#Carga y Acción Manual (Celda de Navegación)
url_test = "https://www.tripadvisor.es/Hotel_Review-g187514-d1192534-Reviews-Hotel_Riu_Plaza_Espana-Madrid.html"
driver.get(url_test)

print("¡ATENCIÓN! Ve a la interfaz gráfica del Docker ahora:")
print("1. Acepta las Cookies manualmente si aparecen.")
print("2. Baja con el scroll hasta que veas las reseñas físicamente.")
print("3. Una vez las veas, regresa aquí y ejecuta la siguiente celda.")

¡ATENCIÓN! Ve a la interfaz gráfica del Docker ahora:
1. Acepta las Cookies manualmente si aparecen.
2. Baja con el scroll hasta que veas las reseñas físicamente.
3. Una vez las veas, regresa aquí y ejecuta la siguiente celda.


In [ ]:
#Extracción de Reseñas (Celda de Datos)
# Hacemos un scroll pequeño para activar el DOM
driver.execute_script("window.scrollBy(0, 500);")
time.sleep(2)

# Buscamos los bloques usando el selector de 'Card' de 2026
bloques = driver.find_elements(By.CSS_SELECTOR, "div[data-test-target='HR_CC_Card']")

if len(bloques) == 0:
    # Intento con selector alternativo si el anterior falla
    bloques = driver.find_elements(By.XPATH, "//div[@data-reviewid]")

print(f"Reseñas encontradas: {len(bloques)}")

resultados = []
for b in bloques:
    try:
        texto = b.find_element(By.CSS_SELECTOR, "span.yY6yc, div.biGQs, q").text
        rating_raw = b.find_element(By.CSS_SELECTOR, "span.ui_bubble_rating").get_attribute("class")
        rating = int(rating_raw.split("_")[-1]) / 10
        
        resultados.append({"Rating": rating, "Comentario": texto[:150]})
    except:
        continue

pd.DataFrame(resultados)